# Anomaly Detection — Ray Parallel + MLflow Tracking

This notebook reuses the `oxy` anomaly-detection functions and parallelises them
across a Ray cluster, with every model run tracked as an MLflow child run.

**Pipeline:**
1. Feature engineering (processor + FeatureFactory) — per-sensor, Ray-parallel
2. Model scoring (run_model functions) — per-sensor × per-model, Ray-parallel
3. Consensus aggregation + cross-sensor analysis
4. All results tracked in MLflow

In [3]:
import sys, os
from pathlib import Path

# ── Detect environment ──
# K8s pods (JupyterHub + Ray): oxy mounted at /mnt/oxy via hostPath
# Local dev: use the source checkout directly
OXY_ROOT = Path(os.environ.get("OXY_ROOT", "/mnt/oxy"))
if not (OXY_ROOT / "modules" / "anomaly-detection" / "src").exists():
    OXY_ROOT = Path("/home/rami/Work/kyper/oxy")

ANOMALY_DIR = OXY_ROOT / "modules" / "anomaly-detection"
sys.path.insert(0, str(ANOMALY_DIR))
print(f"OXY_ROOT:    {OXY_ROOT}")
print(f"ANOMALY_DIR: {ANOMALY_DIR}")

OXY_ROOT:    /home/rami/Work/kyper/oxy
ANOMALY_DIR: /home/rami/Work/kyper/oxy/modules/anomaly-detection


In [4]:
import yaml
import pandas as pd
import numpy as np
import ray
import mlflow

# Reuse oxy functions directly — no modifications
from src.feature_factory import FeatureFactory
from src.visualizer import save_diagnostic
from src.cross_sensor import run_cross_sensor
from src.feature_importance import compute_and_save as save_feature_importance
from algorithms.copod import run_model as copod_run
from algorithms.ecod import run_model as ecod_run
from algorithms.lof import run_model as lof_run
from algorithms.isolation_forest import run_model as iforest_run
from algorithms.loda import run_model as loda_run
from algorithms.hbos import run_model as hbos_run

In [3]:
MODEL_REGISTRY = {
    "copod":            copod_run,
    "ecod":             ecod_run,
    "lof":              lof_run,
    "isolation_forest": iforest_run,
    "loda":             loda_run,
    "hbos":             hbos_run,
}

## 1. Configuration

In [6]:
# Load config from oxy
with open(ANOMALY_DIR / "config.yaml") as f:
    cfg = yaml.safe_load(f)

DATA_DIR = (ANOMALY_DIR / cfg["data"]["root"]).resolve()
ACTIVE_MODELS = cfg["active_models"]

# In K8s the hostPath mount is read-only — write features/results to home dir
if os.environ.get("OXY_ROOT"):
    FEATURES_DIR = Path.home() / "results" / "anomaly-detection" / "features"
    RESULTS_DIR  = Path.home() / "results" / "anomaly-detection" / "detections"
else:
    FEATURES_DIR = (ANOMALY_DIR / cfg["data"]["features"]).resolve()
    RESULTS_DIR  = (ANOMALY_DIR / cfg["data"]["results"]).resolve()

FEATURES_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Data root:     {DATA_DIR}")
print(f"Features dir:  {FEATURES_DIR}")
print(f"Results dir:   {RESULTS_DIR}")
print(f"Active models: {[m['name'] for m in ACTIVE_MODELS]}")

Data root:     /home/rami/Work/kyper/oxy/data
Features dir:  /home/rami/Work/kyper/oxy/results/anomaly-detection/features
Results dir:   /home/rami/Work/kyper/oxy/results/anomaly-detection/detections
Active models: ['copod', 'ecod', 'lof', 'isolation_forest', 'loda', 'hbos']


## 2. Connect to Ray & MLflow

In [ ]:
import os
import atexit

# Shared pip deps needed by Ray workers for anomaly-detection tasks
RUNTIME_PIP = [
    "pandas", "numpy", "scikit-learn", "pyod",
    "matplotlib", "xgboost", "statsmodels", "pyyaml", "mlflow>=2.0",
]

# Disconnect any stale Ray session from a previous kernel run
if ray.is_initialized():
    ray.shutdown()

# Ray — connect to the cluster (RAY_ADDRESS set by JupyterHub pre_spawn_hook)
# Falls back to local for development outside the cluster
ray_address = os.environ.get("RAY_ADDRESS", None)
if ray_address:
    # Ship oxy source code + pip deps to Ray workers via runtime_env
    RUNTIME_ENV = {
        "pip": RUNTIME_PIP,
        "working_dir": str(ANOMALY_DIR),
        "excludes": ["__pycache__", "*.pyc"],
    }
    ray.init(address=ray_address, runtime_env=RUNTIME_ENV, ignore_reinit_error=True)
else:
    ray.init(
    ignore_reinit_error=True,
    runtime_env={"working_dir": str(ANOMALY_DIR)},
    )

# Ensure Ray shuts down cleanly on kernel restart/exit
atexit.register(ray.shutdown)

print(f"Ray cluster: {ray.cluster_resources()}")

# MLflow — tracking URI set by JupyterHub env, or local file store for development
mlflow_uri = os.environ.get("MLFLOW_TRACKING_URI", None)
if mlflow_uri is None:
    # Local dev: use file-based store (no server required)
    local_mlflow_dir = Path.home() / "mlflow-runs"
    local_mlflow_dir.mkdir(exist_ok=True)
    mlflow_uri = str(local_mlflow_dir)
mlflow.set_tracking_uri(mlflow_uri)
print(f"MLflow tracking: {mlflow_uri}")

2026-04-15 19:21:16,528	INFO worker.py:1832 -- Started a local Ray instance. View the dashboard at http://127.0.0.1:8265 


Ray cluster: {'node:172.19.119.250': 1.0, 'node:__internal_head__': 1.0, 'GPU': 1.0, 'object_store_memory': 14610809241.0, 'memory': 29221618484.0, 'accelerator_type:RTX': 1.0, 'CPU': 32.0}
MLflow tracking: /home/rami/mlflow-runs


## 3. Feature Engineering — Ray Parallel

Discover raw sensor CSVs, generate features in parallel using `FeatureFactory`.

In [8]:
# Discover raw sensor CSVs (same filtering as processor.py)
EXCLUDE = {"hust", "NAB", "downtime"}
raw_csvs = sorted(
    p for p in DATA_DIR.rglob("*.csv")
    if not any(part in EXCLUDE for part in p.parts)
)
print(f"Found {len(raw_csvs)} raw sensor CSVs")

Found 282 raw sensor CSVs


In [9]:
@ray.remote
def build_features(raw_path_str: str, data_dir_str: str, features_dir_str: str) -> str:
    """Generate features for a single sensor CSV. Returns the feature file path."""
    raw_path = Path(raw_path_str)
    data_dir = Path(data_dir_str)
    features_dir = Path(features_dir_str)

    rel = raw_path.relative_to(data_dir)
    feature_path = features_dir / rel

    # Skip if features are already up-to-date
    if feature_path.exists() and feature_path.stat().st_mtime > raw_path.stat().st_mtime:
        return str(feature_path)

    df = pd.read_csv(raw_path, index_col=0, parse_dates=True)
    if df.shape[1] != 1:
        return ""  # skip multi-column files

    factory = FeatureFactory()
    features = factory.transform(df)
    feature_path.parent.mkdir(parents=True, exist_ok=True)
    features.to_csv(feature_path)
    return str(feature_path)

In [10]:
%%time
# Launch feature engineering in parallel
feature_futures = [
    build_features.remote(str(p), str(DATA_DIR), str(FEATURES_DIR))
    for p in raw_csvs
]
feature_paths = ray.get(feature_futures)
feature_paths = [p for p in feature_paths if p]  # drop empties
print(f"Feature files ready: {len(feature_paths)}")

CPU times: user 59.9 ms, sys: 25.6 ms, total: 85.5 ms
Wall time: 1.08 s


RayTaskError(RuntimeError): [36mray::build_features()[39m (pid=7551, ip=172.19.119.250)
RuntimeError: The remote function failed to import on the worker. This may be because needed library dependencies are not installed in the worker environment or cannot be found from sys.path ['/home/rami/Work/kyper/kyper-framework/notebooks', '/home/rami/Work/kyper/.venv/lib/python3.12/site-packages/ray/thirdparty_files', '/home/rami/Work/kyper/.venv/lib/python3.12/site-packages/ray/_private/workers', '/home/rami/.local/share/uv/python/cpython-3.12.11-linux-x86_64-gnu/lib/python312.zip', '/home/rami/.local/share/uv/python/cpython-3.12.11-linux-x86_64-gnu/lib/python3.12', '/home/rami/.local/share/uv/python/cpython-3.12.11-linux-x86_64-gnu/lib/python3.12/lib-dynload', '/home/rami/Work/kyper/.venv/lib/python3.12/site-packages']:

[36mray::build_features()[39m (pid=7551, ip=172.19.119.250)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
ModuleNotFoundError: No module named 'src'

2026-04-15 19:21:37,956	ERROR worker.py:422 -- Unhandled error (suppress with 'RAY_IGNORE_UNHANDLED_ERRORS=1'): ray::build_features() (pid=7522, ip=172.19.119.250)
RuntimeError: The remote function failed to import on the worker. This may be because needed library dependencies are not installed in the worker environment or cannot be found from sys.path ['/home/rami/Work/kyper/kyper-framework/notebooks', '/home/rami/Work/kyper/.venv/lib/python3.12/site-packages/ray/thirdparty_files', '/home/rami/Work/kyper/.venv/lib/python3.12/site-packages/ray/_private/workers', '/home/rami/.local/share/uv/python/cpython-3.12.11-linux-x86_64-gnu/lib/python312.zip', '/home/rami/.local/share/uv/python/cpython-3.12.11-linux-x86_64-gnu/lib/python3.12', '/home/rami/.local/share/uv/python/cpython-3.12.11-linux-x86_64-gnu/lib/python3.12/lib-dynload', '/home/rami/Work/kyper/.venv/lib/python3.12/site-packages']:

ray::build_features() (pid=7522, ip=172.19.119.250)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^

## 4. Anomaly Detection — Ray Parallel + MLflow Tracking

For each sensor, run all 6 models in parallel. Each (sensor, model) pair
is a Ray task that logs to MLflow as a nested child run.

In [ ]:
@ray.remote
def run_single_model(
    feature_path_str: str,
    model_name: str,
    model_kwargs: dict,
    mlflow_uri: str,
    experiment_name: str,
    parent_run_id: str,
) -> dict:
    """
    Run one model on one sensor CSV.
    Logs params, metrics to MLflow as a nested run.
    Returns {sensor, model, n_anomalies, scores, labels}.
    """
    import mlflow as _mlflow
    _mlflow.set_tracking_uri(mlflow_uri)
    _mlflow.set_experiment(experiment_name)

    feature_path = Path(feature_path_str)
    sensor = feature_path.stem
    machine = str(feature_path.parent.name)

    X_df = pd.read_csv(feature_path, index_col=0, parse_dates=True)

    # Dispatch to the correct model
    model_fn = {
        "copod": copod_run, "ecod": ecod_run, "lof": lof_run,
        "isolation_forest": iforest_run, "loda": loda_run, "hbos": hbos_run,
    }[model_name]

    scores, labels = model_fn(X_df, **model_kwargs)
    n_anomalies = int(labels.sum())
    anomaly_rate = float(labels.mean())

    # Log to MLflow as a child run
    with _mlflow.start_run(run_name=f"{sensor}_{model_name}", nested=True, parent_run_id=parent_run_id):
        _mlflow.log_params({
            "sensor": sensor,
            "machine": machine,
            "model": model_name,
            **{k: str(v) for k, v in model_kwargs.items()},
        })
        _mlflow.log_metrics({
            "n_anomalies": n_anomalies,
            "anomaly_rate": anomaly_rate,
            "n_samples": len(X_df),
            "score_mean": float(scores.mean()),
            "score_std": float(scores.std()),
        })

    return {
        "feature_path": feature_path_str,
        "sensor": sensor,
        "model": model_name,
        "n_anomalies": n_anomalies,
        "anomaly_rate": anomaly_rate,
        "scores": scores.tolist(),
        "labels": labels.tolist(),
    }

In [ ]:
feature_csvs = sorted(FEATURES_DIR.rglob("*.csv"))
print(f"Feature CSVs to process: {len(feature_csvs)}")
print(f"Models per sensor: {len(ACTIVE_MODELS)}")
print(f"Total tasks: {len(feature_csvs) * len(ACTIVE_MODELS)}")

In [ ]:
%%time
EXPERIMENT_NAME = "anomaly-detection-parallel"
mlflow.set_experiment(EXPERIMENT_NAME)

with mlflow.start_run(run_name="parallel-run-all") as parent_run:
    parent_run_id = parent_run.info.run_id

    # Log top-level config
    mlflow.log_params({
        "n_sensors": len(feature_csvs),
        "n_models": len(ACTIVE_MODELS),
        "total_tasks": len(feature_csvs) * len(ACTIVE_MODELS),
        "model_names": ",".join(m["name"] for m in ACTIVE_MODELS),
    })

    # Launch all (sensor × model) tasks
    futures = []
    for fpath in feature_csvs:
        for model_cfg in ACTIVE_MODELS:
            model_name = model_cfg["name"]
            kwargs = {k: v for k, v in model_cfg.items() if k != "name"}
            futures.append(
                run_single_model.remote(
                    str(fpath), model_name, kwargs,
                    mlflow_uri, EXPERIMENT_NAME, parent_run_id,
                )
            )

    print(f"Submitted {len(futures)} Ray tasks...")
    results = ray.get(futures)
    print(f"Completed {len(results)} tasks.")

    # Log summary metrics to parent run
    total_anomalies = sum(r["n_anomalies"] for r in results)
    mean_rate = np.mean([r["anomaly_rate"] for r in results])
    mlflow.log_metrics({
        "total_anomalies": total_anomalies,
        "mean_anomaly_rate": mean_rate,
    })

## 5. Assemble Results & Generate Diagnostics

In [ ]:
from collections import defaultdict

# Group results by feature_path
sensor_results = defaultdict(list)
for r in results:
    sensor_results[r["feature_path"]].append(r)

model_names = [m["name"] for m in ACTIVE_MODELS]

for feature_path_str, model_results in sensor_results.items():
    feature_path = Path(feature_path_str)
    sensor = model_results[0]["sensor"]
    X_df = pd.read_csv(feature_path, index_col=0, parse_dates=True)
    results_df = X_df.copy()

    # Attach each model's scores & labels
    for mr in model_results:
        results_df[f"{mr['model']}_score"] = mr["scores"]
        results_df[f"{mr['model']}_label"] = mr["labels"]

    # Consensus column
    label_cols = [f"{m}_label" for m in model_names]
    results_df["consensus"] = results_df[label_cols].sum(axis=1).astype(int)

    # Save to results dir — mirror the feature directory structure
    rel = feature_path.relative_to(FEATURES_DIR)
    out_dir = RESULTS_DIR / rel.parent
    out_dir.mkdir(parents=True, exist_ok=True)

    results_df.to_csv(out_dir / f"{sensor}_results.csv")
    save_diagnostic(results_df, model_names, sensor, out_dir / f"{sensor}_diagnostic.png")

print(f"Assembled results for {len(sensor_results)} sensors.")

## 6. Cross-Sensor Analysis

Run cross-sensor consensus for each machine directory.

In [ ]:
# Get unique machine directories from results
machine_dirs = set()
for fpath_str in sensor_results:
    fpath = Path(fpath_str)
    rel = fpath.relative_to(FEATURES_DIR)
    machine_dir = RESULTS_DIR / rel.parent
    machine_dirs.add(machine_dir)

for mdir in sorted(machine_dirs):
    result_csvs = list(mdir.glob("*_results.csv"))
    if result_csvs:
        print(f"  Cross-sensor: {mdir.relative_to(RESULTS_DIR)} ({len(result_csvs)} sensors)")
        run_cross_sensor(mdir)

## 7. Summary

In [ ]:
# Build summary DataFrame
summary_rows = []
for r in results:
    summary_rows.append({
        "sensor": r["sensor"],
        "model": r["model"],
        "n_anomalies": r["n_anomalies"],
        "anomaly_rate": round(r["anomaly_rate"], 4),
    })

summary_df = pd.DataFrame(summary_rows)
print(f"\nTotal tasks: {len(summary_df)}")
print(f"Total anomalies: {summary_df['n_anomalies'].sum()}")
print(f"\nPer-model anomaly counts:")
print(summary_df.groupby("model")["n_anomalies"].sum().to_string())
print(f"\nMLflow parent run: {parent_run_id}")
print(f"MLflow UI: {mlflow_uri}/#/experiments")

In [ ]:
# Top sensors by anomaly rate (across all models)
pivot = summary_df.pivot_table(index="sensor", columns="model", values="anomaly_rate")
pivot["mean_rate"] = pivot.mean(axis=1)
pivot.sort_values("mean_rate", ascending=False).head(15)

In [ ]:
ray.shutdown()